# DSCI 552: HW3
- **Name:** Pavanitha Manche
- **USC ID:** 2866839249
- **GitHub Username:** Pavanitha09

## 1. Time Series Classification Part 1: Feature Creation/Extraction

In [15]:
import os
import re
import numpy as np
import pandas as pd
from scipy.stats import bootstrap

### 1(a) Data Loading

In [16]:
data_path = '../data/AReM/'
COLUMNS = ['time', 'avg_rss12', 'var_rss12', 'avg_rss13', 'var_rss13', 'avg_rss23', 'var_rss23']
ACTIVITIES = ['bending1', 'bending2', 'cycling', 'lying', 'sitting', 'standing', 'walking']

def read_instance(path):
    # Comment lines start with '#'; separators may be commas or spaces; drop any empty trailing column
    df = pd.read_csv(path, comment='#', header=None, names=range(8), sep=r'[,\s]+', engine='python')
    df = df.iloc[:, :7]
    df.columns = COLUMNS
    return df

def dataset_number(file_name):
    return int(re.search(r'\d+', file_name).group())

instances = []
for activity in ACTIVITIES:
    files = sorted(os.listdir(os.path.join(data_path, activity)), key=dataset_number)
    for file_name in files:
        instances.append({
            'activity': activity,
            'dataset': dataset_number(file_name),
            'data': read_instance(os.path.join(data_path, activity, file_name)),
        })

print(f"Loaded {len(instances)} instances")
display(pd.Series([inst['activity'] for inst in instances]).value_counts().rename('instances').to_frame())
display(instances[0]['data'].head())

Loaded 88 instances


,instances
cycling,15
lying,15
sitting,15
standing,15
walking,15
bending1,7
bending2,6


,time,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23
0,0,39.25,0.43,22.75,0.43,33.75,1.3
1,250,39.25,0.43,23.00,0.00,33.00,0.0
2,500,39.25,0.43,23.25,0.43,33.00,0.0
3,750,39.50,0.50,23.00,0.71,33.00,0.0
4,1000,39.50,0.50,24.00,0.00,33.00,0.0


### 1(b) Train / Test Split

- **Test data:** datasets 1 and 2 from `bending1` and `bending2`, and datasets 1, 2 and 3 from every other folder.
- **Train data:** all remaining datasets.

In [18]:
def is_test(activity, dataset):
    if activity in ('bending1', 'bending2'):
        return dataset in (1, 2)
    return dataset in (1, 2, 3)

for inst in instances:
    inst['split'] = 'test' if is_test(inst['activity'], inst['dataset']) else 'train'

split_counts = pd.crosstab(
    pd.Series([inst['activity'] for inst in instances], name='activity'),
    pd.Series([inst['split'] for inst in instances], name='split'),
    margins=True,
)
display(split_counts)

split,test,train,All
activity,,,
bending1,2,5,7
bending2,2,4,6
cycling,3,12,15
lying,3,12,15
sitting,3,12,15
standing,3,12,15
walking,3,12,15
All,19,69,88


### 1(c) Feature Extraction

#### i. Time-Domain Features Used in Time Series Classification


| # | Feature | Description |
| :-: | :-- | :-- |
| 1 | **Minimum** | The smallest value in the series. |
| 2 | **Maximum** | The largest value in the series. |
| 3 | **Mean** | The arithmetic average of all values; the typical level of the series. |
| 4 | **Median** | The middle value (50th percentile); less affected by outliers than the mean. |
| 5 | **Standard deviation** | How spread out the values are around the mean. |
| 6 | **First quartile (Q1)** | The 25th percentile; 25% of the values lie below it. |
| 7 | **Third quartile (Q3)** | The 75th percentile; 75% of the values lie below it. |
| 8 | **Interquartile range (IQR)** | Q3 − Q1; the spread of the middle 50% of the values. |
| 9 | **Skewness** | How asymmetric the distribution of values is (a longer tail on one side). |
| 10 | **Kurtosis** | How heavy the tails of the distribution are, i.e. how often extreme values occur. |
| 11 | **Auto-correlation** | The correlation between a series and a time-shifted (lagged) copy of itself; shows repeating patterns. |
| 12 | **Cross-correlation** | The correlation between two different series at various lags; shows how similar they are. |

#### ii. Extracting the Features



In [19]:
SERIES = COLUMNS[1:]
FEATURES = {
    'min': lambda s: s.min(),
    'max': lambda s: s.max(),
    'mean': lambda s: s.mean(),
    'median': lambda s: s.median(),
    'std': lambda s: s.std(),
    '1st_quart': lambda s: s.quantile(0.25),
    '3rd_quart': lambda s: s.quantile(0.75),
}

def extract_features(series_df):
    row = {}
    for i, col in enumerate(SERIES, start=1):
        for name, func in FEATURES.items():
            row[f'{name}{i}'] = func(series_df[col])
    return row

rows = []
for inst in instances:
    row = extract_features(inst['data'])
    row['activity'] = inst['activity']
    row['dataset'] = inst['dataset']
    row['split'] = inst['split']
    rows.append(row)

features_df = pd.DataFrame(rows)
features_df.index = pd.RangeIndex(1, len(features_df) + 1, name='Instance')
feature_cols = [c for c in features_df.columns if c not in ('activity', 'dataset', 'split')]

print(f"Feature table shape: {features_df.shape}  ({len(feature_cols)} features + activity, dataset, split)")
display(features_df)

Feature table shape: (88, 45)  (42 features + activity, dataset, split)


,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,mean2,...,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6,activity,dataset,split
Instance,,,,,,,,,,,,,,,,,,,,,
1,37.25,45.00,40.624792,40.50,1.476967,39.25,42.0000,0.0,1.30,0.358604,...,0.00,1.92,0.570583,0.43,0.582915,0.0000,1.3000,bending1,1,test
2,38.00,45.67,42.812812,42.50,1.435550,42.00,43.6700,0.0,1.22,0.372437,...,0.00,3.11,0.571083,0.43,0.601010,0.0000,1.3000,bending1,2,test
3,35.00,47.40,43.954500,44.33,1.558835,43.00,45.0000,0.0,1.70,0.426250,...,0.00,1.79,0.493292,0.43,0.513506,0.0000,0.9400,bending1,3,train
4,33.00,47.75,42.179812,43.50,3.670666,39.15,45.0000,0.0,3.00,0.696042,...,0.00,2.18,0.613521,0.50,0.524317,0.0000,1.0000,bending1,4,train
5,33.00,45.75,41.678063,41.75,2.243490,41.33,42.7500,0.0,2.83,0.535979,...,0.00,1.79,0.383292,0.43,0.389164,0.0000,0.5000,bending1,5,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,19.50,45.33,33.586875,34.25,4.650935,30.25,37.0000,0.0,14.67,4.576562,...,0.00,8.32,3.259729,3.11,1.640243,2.0500,4.3225,walking,11,train
85,19.75,45.50,34.322750,35.25,4.752477,31.00,38.0000,0.0,13.47,4.456333,...,0.00,9.67,3.432563,3.20,1.732727,2.1575,4.5650,walking,12,train
86,19.50,46.00,34.546229,35.25,4.842294,31.25,37.8125,0.0,12.47,4.371958,...,0.00,10.00,3.338125,3.08,1.656742,2.1600,4.3350,walking,13,train


In [20]:
train_df = features_df[features_df['split'] == 'train']
test_df = features_df[features_df['split'] == 'test']
print(f"Train instances: {len(train_df)}, Test instances: {len(test_df)}")

Train instances: 69, Test instances: 19


#### iii. Standard Deviation of Each Feature and 90% Bootstrap Confidence Interval

In [21]:
def sample_std(x, axis=-1):
    return np.std(x, ddof=1, axis=axis)

ci_rows = []
for col in feature_cols:
    values = features_df[col].to_numpy()
    res = bootstrap((values,), sample_std, confidence_level=0.90, n_resamples=10000,
                    method='percentile', random_state=42)
    ci_rows.append({
        'Feature': col,
        'Std': sample_std(values),
        'CI Lower (5%)': res.confidence_interval.low,
        'CI Upper (95%)': res.confidence_interval.high,
    })

std_ci_df = pd.DataFrame(ci_rows).set_index('Feature')
pd.set_option('display.max_rows', 50)
display(std_ci_df.round(4))

,Std,CI Lower (5%),CI Upper (95%)
Feature,,,
min1,9.5700,8.2672,10.7818
max1,4.3944,3.3508,5.2999
mean1,5.3357,4.7043,5.8894
median1,5.4401,4.7869,6.0092
std1,1.7722,1.5651,1.9503
1st_quart1,6.1536,5.5543,6.6536
3rd_quart1,5.1389,4.3302,5.8448
min2,0.0000,0.0000,0.0000
max2,5.0627,4.6221,5.4023


#### iv. Selecting the Three Most Important Features

**Chosen features: mean, standard deviation and maximum.**

- **Mean** - the typical level of a series. It separates the activities best on its own (posture in the `avg` series, amount of movement in the `var` series).
- **Standard deviation** - how much the series varies. It is the least correlated with the other features, so it adds information they do not have.
- **Maximum** - the strongest bursts of movement, which clearly distinguish walking and cycling from the still activities.

## 2. ISLR 3.7.4: Linear vs. Cubic Regression (Training and Test RSS)

### (a) Linear - Training RSS

**The cubic regression will have a training RSS lower than or equal to the linear regression.**

The linear model is a special case of the cubic model (set $\beta_2 = \beta_3 = 0$). Least squares picks the coefficients that minimise RSS on the training data, so the cubic fit can always do at least as well as the linear fit. In practice the extra terms fit some of the random noise $\epsilon$, so its training RSS will be slightly lower.

### (b) Linear - Test RSS

**The linear regression is expected to have the lower test RSS.**

The linear model matches the true relationship, so it has no bias. The cubic model's extra terms only fit noise in the training data (overfitting), which increases variance without reducing bias. That noise does not repeat in new data, so the cubic model usually predicts the test data worse. This is expected, not guaranteed, because with a small sample the result can vary.

### (c) Non-linear - Training RSS

**The cubic regression will have a training RSS lower than or equal to the linear regression.**

The reasoning is the same as in (a): the cubic model contains the linear model, so on training data it can never fit worse. With a non-linear truth the cubic model can also follow the curvature, so the gap is usually larger.

### (d) Non-linear - Test RSS

**There is not enough information to tell.**

It depends on how far the true relationship is from linear:
- If it is **far from linear**, the linear model has high bias and the cubic model will most likely have the lower test RSS.
- If it is **only slightly non-linear**, the linear model's small bias may cost less than the cubic model's extra variance, and the linear model could have the lower test RSS.